In [ ]:
# ═══════════════════════════════════════════════════════════
#  EEG_11 — DHSLP / DHSLF: Dynamic Hypergraph Semi-supervised
#            Learning within Projected / Selected Feature Subspace
#  Paper: Li et al. 2025, J. Neural Engineering
#  DOI: 10.1088/1741-2552/adeec8
# ═══════════════════════════════════════════════════════════
#
#  Differenza chiave rispetto a EEG_10:
#   - Vertici = TRIAL (non canali)
#   - Input   = Feature DE (Differential Entropy) per banda
#   - Ottimizzazione iterativa ADMM (non gradient descent)
#   - Semi-supervised: usa labeled + unlabeled insieme
#
USE_CLUSTERS   = True
CLUSTER_SCHEME = "concr4"   # 4 classi semantiche
#
# Iperparametri DHSLP/DHSLF (paper default)
XI         = 0.5    # soglia iperspigolo: d(vi,vj) <= xi * mean_dist(vi)
ALPHA      = 0.5    # smoothness feature subspace
BETA       = 0.01   # regularizzazione pesi iperspigoli
F_DIM      = 20     # dimensione subspace proiettato (DHSLP)
MAX_ITER   = 50     # iterazioni massime ADMM
# Modalità valutazione: within-subject (più fedele al paper)
# Split: 80% labeled, 20% unlabeled per soggetto
LABEL_RATIO = 0.8
SEED = 42
# ═══════════════════════════════════════════════════════════

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import h5py
import scipy.signal as signal
import scipy.linalg as linalg
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

print("numpy:", np.__version__)
print("scipy:", __import__("scipy").__version__)

In [ ]:
# ============================================================
# CONFIGURAZIONE PATHS
# ============================================================

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)

META_CSV    = project_root / "data" / "interim" / "eeg_metadata.csv"
ELOC_PATH   = project_root / "src" / "io" / "ebneuro.locs"
interim_dir = project_root / "data" / "interim"

N_CHANS = 59
N_TIMES = 384
SFREQ   = 256

# Bande di frequenza per DE feature
FREQ_BANDS = {
    "delta": (1,  4),
    "theta": (4,  8),
    "alpha": (8,  13),
    "beta":  (13, 30),
    "gamma": (30, 50),
}
N_BANDS = len(FREQ_BANDS)
D_FEAT  = N_CHANS * N_BANDS   # dimensione feature vector: 59 × 5 = 295

RESULTS_CSV = project_root / "data" / "interim" / "eeg11_dhslp_results.csv"

print(f"project_root: {project_root}")
print(f"Feature dim:  D = {D_FEAT}  ({N_CHANS} canali × {N_BANDS} bande)")

In [ ]:
# ============================================================
# METADATA + LABEL SCHEME
# ============================================================

import sys
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

meta = pd.read_csv(META_CSV)
meta = meta[~(
    (meta["path_h5"].str.contains("08_05.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_01.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_03.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_04.h5") & (meta["epoch_idx"] >= 34))
)]
meta["subject_id"] = meta["subject_id"].astype(str).str.zfill(2)

def read_eloc_names(path):
    names = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]

ch_names_61 = read_eloc_names(ELOC_PATH)
keep_idx    = [i for i, n in enumerate(ch_names_61) if n not in {"A1", "A2"}]
assert len(keep_idx) == N_CHANS

_scheme = CLUSTER_SCHEME if USE_CLUSTERS else "raw110"
labelid2cluster, N_CLASSES, cluster_names = load_label_scheme(_scheme, interim_dir)

print(f"Schema: {CLUSTER_SCHEME}, Classi: {N_CLASSES}, Chance: {100/N_CLASSES:.1f}%")
print(f"Soggetti unici: {meta['subject_id'].nunique()}")
print(f"Totale record:  {len(meta)}")

In [ ]:
# ============================================================
# ESTRAZIONE FEATURE: DIFFERENTIAL ENTROPY (DE)
# ============================================================
#
# DE per una banda di frequenza banda b su canale c:
#   DE(x_b) = log(2πe · σ²_b)  [approssimazione Gaussiana]
#          ∝  log(σ²_b)  [costante eliminabile]
#
# È la feature standard per EEG emotion/speech recognition
# (Shi et al. 2013; usata implicitamente in DHSLP).
# Dimensione output: [n_trials, N_CHANS × N_BANDS] = [n, 295]
# ============================================================


def bandpass_filter(x, low, high, fs=SFREQ, order=5):
    """Butterworth bandpass su segnale [channels, samples]."""
    nyq = fs / 2.0
    b, a = signal.butter(order, [low / nyq, high / nyq], btype="band")
    return signal.filtfilt(b, a, x, axis=1)


def compute_de_features(epoch_data):
    """
    Calcola feature DE per un singolo trial.
    Input:  epoch_data [N_CHANS, N_TIMES]
    Output: feature_vec [N_CHANS × N_BANDS]
    """
    feats = []
    for (low, high) in FREQ_BANDS.values():
        try:
            filtered = bandpass_filter(epoch_data, low, high)
        except Exception:
            filtered = epoch_data  # fallback
        # Log-varianza per ogni canale
        var = np.var(filtered, axis=1).clip(1e-10)
        feats.append(np.log(var))
    return np.concatenate(feats)   # [N_CHANS × N_BANDS]


def extract_de_for_subject(records, keep_idx):
    """
    Estrae feature DE per tutti i trial di un soggetto.
    Returns: X [n_trials, D_FEAT], y [n_trials]
    """
    paths_map = defaultdict(list)
    for i, r in enumerate(records):
        paths_map[r["path_h5"]].append((i, int(r["epoch_idx"]), int(r["label_idx"])))

    X_list = [None] * len(records)
    y_list = [None] * len(records)

    for path, items in paths_map.items():
        with h5py.File(path, "r") as f:
            for (i, epoch_idx, label_idx) in items:
                raw = f["data"][epoch_idx][keep_idx, :].astype(np.float32)
                X_list[i] = compute_de_features(raw)
                y_list[i] = labelid2cluster[label_idx]

    return np.stack(X_list), np.array(y_list)


# Test su un soggetto campione
_test_rec = meta[meta["subject_id"] == "00"][["path_h5", "epoch_idx", "label_idx"]].to_dict("records")
_X, _y = extract_de_for_subject(_test_rec[:5], keep_idx)
print(f"DE feature shape: {_X.shape}  (5 trial × {D_FEAT} feature)")
print(f"feature range: [{_X.min():.2f}, {_X.max():.2f}]")
print("DE estrazione OK")

In [ ]:
# ============================================================
# DHSLP / DHSLF — IMPLEMENTAZIONE FEDELE AL PAPER
# Li et al. 2025, Algorithm 1
# ============================================================
#
# Input:
#   X_l [l, d]  — feature labeled (training)
#   Y_l [l, c]  — one-hot label matrix labeled
#   X_u [u, d]  — feature unlabeled (test)
#   model       — "DHSLP" (proiezione) o "DHSLF" (feature selection)
#
# Output:
#   F_u [u, c]  — predicted label matrix per unlabeled samples
#
# Algoritmo iterativo:
#   while not converged:
#     1. Calcola D_v, D_e, L (Laplaciano ipergraph)
#     2. Update F_u = -L_uu^{-1} L_ul Y_l
#     3. DHSLP: Update M (primi f eigenvettori di αX^T L X)
#        DHSLF: Update Θ (theta_i = 1/t_i / sum(1/t_j), t_i = x_i^T L x_i)
#     4. Update H, U (distanza in subspace trasformato)
#     5. Update W (Newton + Lagrange)
# ============================================================


def build_incidence_matrix(X_transformed, xi=0.5):
    """
    Costruisce H [n, n] e U [n] tramite approccio distance-based.
    
    Per ogni vertice v_i: e_i = {v_j | d(v_i,v_j) <= xi * mean_dist(v_i)}
    Vertex weight: U(v_i) = d̄_i / sum(d̄_j)
    
    Input: X_transformed [n, f]  — features nel subspace corrente
    Output: H [n, n], U [n]
    """
    n = X_transformed.shape[0]

    # Distanze euclidee a coppie
    diff = X_transformed[:, np.newaxis, :] - X_transformed[np.newaxis, :, :]
    D_mat = np.sqrt((diff ** 2).sum(axis=2))  # [n, n]

    # Mean distance per vertice
    d_mean = D_mat.mean(axis=1)  # [n]

    # Incidence matrix H: H[i,j]=1 se v_i ∈ e_j (iperspigolo centrato su v_j)
    H = (D_mat <= xi * d_mean[np.newaxis, :]).astype(np.float64)  # [n, n]

    # Vertex weights: inversamente proporzionali alla distanza media
    d_mean_safe = d_mean.clip(1e-10)
    U_vec = d_mean_safe / d_mean_safe.sum()  # [n]

    return H, U_vec


def compute_laplacian(H, W_vec, U_vec):
    """
    L = U - D_v^{-1/2} U H W D_e^{-1} H^T U D_v^{-1/2}
    
    H [n, m], W_vec [m], U_vec [n]
    """
    n = H.shape[0]
    U_diag = np.diag(U_vec)
    W_diag = np.diag(W_vec)

    # D_v = diag(H W 1)
    Dv_vec  = H @ W_vec               # [n] — vertex degrees
    Dv_vec  = Dv_vec.clip(1e-10)
    Dv_inv_sqrt = np.diag(Dv_vec ** (-0.5))

    # D_e = diag(H^T U 1)  — hyperedge degrees
    De_vec  = H.T @ U_vec             # [m]
    De_vec  = De_vec.clip(1e-10)
    De_inv  = np.diag(De_vec ** (-1.0))

    # Similarity (theta) matrix: S = D_v^{-1/2} U H W D_e^{-1} H^T U D_v^{-1/2}
    Theta_mat = Dv_inv_sqrt @ U_diag @ H @ W_diag @ De_inv @ H.T @ U_diag @ Dv_inv_sqrt

    L = U_diag - Theta_mat
    return L


def dhslp(X_l, Y_l, X_u, xi=XI, alpha=ALPHA, beta=BETA,
          f_dim=F_DIM, max_iter=MAX_ITER, model="DHSLP", verbose=False):
    """
    DHSLP / DHSLF — Algorithm 1 di Li et al. 2025.

    Args:
        X_l:      [l, d]  feature labeled
        Y_l:      [l, c]  one-hot labels
        X_u:      [u, d]  feature unlabeled
        xi:       soglia iperspigolo
        alpha:    smoothness nel subspace
        beta:     regolarizzazione W
        f_dim:    dimensione subspace (solo DHSLP)
        max_iter: iterazioni massime
        model:    "DHSLP" o "DHSLF"

    Returns:
        F_u:  [u, c]  predicted label matrix
        info: dict con convergenza
    """
    l, d = X_l.shape
    u    = X_u.shape[0]
    n    = l + u
    c    = Y_l.shape[1]

    # Concatena labeled + unlabeled
    X = np.vstack([X_l, X_u])   # [n, d]

    # Normalizzazione per stabilità numerica
    X_mean = X.mean(axis=0)
    X_std  = X.std(axis=0).clip(1e-6)
    X = (X - X_mean) / X_std

    # Inizializzazione: subspace = identità (primissima H costruita su X originale)
    if model == "DHSLP":
        M = np.eye(d, min(f_dim, d))         # [d, f]
        X_proj = X @ M                        # [n, f]
    else:  # DHSLF
        theta = np.ones(d) / d               # [d] — pesi feature uniformi
        X_proj = X * theta[np.newaxis, :]    # [n, d]

    # Costruzione iniziale H, U
    H, U_vec = build_incidence_matrix(X_proj, xi=xi)   # H [n,n], U [n]
    m   = n  # numero iperspigoli = numero vertici
    W_vec = np.ones(m) / m                             # W uniforme

    F_u_prev = np.zeros((u, c))
    info = {"obj": [], "converged": False}

    for it in range(max_iter):

        # ── Step 1: Laplaciano ──────────────────────────────
        L = compute_laplacian(H, W_vec, U_vec)   # [n, n]

        # ── Step 2: Update F_u ─────────────────────────────
        # F_u = -L_uu^{-1} L_ul Y_l    (eq. 14)
        L_uu = L[l:, l:]   # [u, u]
        L_ul = L[l:, :l]   # [u, l]
        try:
            F_u = -np.linalg.solve(L_uu + 1e-6 * np.eye(u), L_ul @ Y_l)
        except np.linalg.LinAlgError:
            F_u = -np.linalg.lstsq(L_uu, L_ul @ Y_l, rcond=None)[0]

        # ── Step 3: Update subspace ─────────────────────────
        T = X.T @ L @ X   # [d, d]  (X^T L X)

        if model == "DHSLP":
            # M = primi f eigenvettori di alpha * T  (eq. 18)
            eigvals, eigvecs = linalg.eigh(alpha * T)
            # Vogliamo i f più piccoli (smoothness minimization)
            M = eigvecs[:, :min(f_dim, d)]    # [d, f]
            X_proj = X @ M                    # [n, f]

        else:  # DHSLF
            # theta_i = 1/t_i / sum(1/t_j)  (eq. 32)
            t_vec = np.diag(T).clip(1e-10)   # [d]
            theta = (1.0 / t_vec) / (1.0 / t_vec).sum()
            X_proj = X * theta[np.newaxis, :]

        # ── Step 4: Update H, U ─────────────────────────────
        H, U_vec = build_incidence_matrix(X_proj, xi=xi)

        # ── Step 5: Update W ────────────────────────────────
        # min_w -p^T w - alpha * q^T w + beta ||w||^2
        # s.t. w^T 1 = 1, w_i > 0   (eq. 24)
        # Soluzione: w_i ∝ (p_i + alpha*q_i) / (2*beta), poi normalizzato
        U_diag = np.diag(U_vec)
        Dv_vec = H @ W_vec
        Dv_vec_safe = Dv_vec.clip(1e-10)
        Dv_inv_sqrt = np.diag(Dv_vec_safe ** (-0.5))
        De_vec = (H.T @ U_vec).clip(1e-10)
        De_inv = np.diag(De_vec ** (-1.0))

        P_mat = Dv_inv_sqrt @ U_diag @ H @ De_inv @ H.T @ U_diag @ Dv_inv_sqrt
        Q_mat = Dv_inv_sqrt @ U_diag @ H @ De_inv @ (X_proj @ X_proj.T if model == "DHSLP"
                                                      else X_proj @ X_proj.T) @ De_inv @ H.T @ U_diag @ Dv_inv_sqrt
        p_vec = np.diag(P_mat)   # [m]
        q_vec = np.diag(Q_mat)   # [m]

        w_unnorm = (p_vec + alpha * q_vec) / (2 * beta + 1e-10)
        w_unnorm = w_unnorm.clip(1e-10)
        W_vec = w_unnorm / w_unnorm.sum()

        # ── Convergenza ─────────────────────────────────────
        delta = np.linalg.norm(F_u - F_u_prev, "fro")
        obj   = np.trace(F_u.T @ L[l:, l:] @ F_u)
        info["obj"].append(float(obj))

        if verbose and (it % 10 == 0 or it < 3):
            print(f"  iter {it:3d} | delta={delta:.4f} | obj={obj:.4f}")

        if delta < 1e-4 and it > 2:
            info["converged"] = True
            if verbose:
                print(f"  Convergenza a iter {it}")
            break

        F_u_prev = F_u.copy()

    return F_u, info


print("DHSLP/DHSLF implementati")
print(f"Feature dim D={D_FEAT}, Subspace dim f={F_DIM}")

In [ ]:
# ============================================================
# VALUTAZIONE WITHIN-SUBJECT
# Per ogni soggetto: 80% labeled → predici 20% unlabeled
# Fedele al setup semi-supervised del paper
# ============================================================

def evaluate_subject(records, keep_idx, model="DHSLP", label_ratio=LABEL_RATIO, seed=SEED):
    """
    Valuta DHSLP/DHSLF su un singolo soggetto.
    Returns: acc, bacc, ys_true, ys_pred
    """
    rng = np.random.RandomState(seed)

    # Estrai DE features
    X, y = extract_de_for_subject(records, keep_idx)
    n = len(y)
    if n < 10:
        return None

    # Split labeled / unlabeled
    n_labeled = int(n * label_ratio)
    idx_perm  = rng.permutation(n)
    idx_l     = idx_perm[:n_labeled]
    idx_u     = idx_perm[n_labeled:]

    X_l, X_u = X[idx_l], X[idx_u]
    y_l, y_u = y[idx_l], y[idx_u]

    # One-hot encoding
    Y_l = np.zeros((len(y_l), N_CLASSES))
    Y_l[np.arange(len(y_l)), y_l] = 1.0

    # DHSLP / DHSLF
    F_u, info = dhslp(X_l, Y_l, X_u, model=model)

    # Predizioni: argmax del label matrix
    y_pred = F_u.argmax(axis=1)

    acc  = accuracy_score(y_u, y_pred)
    bacc = balanced_accuracy_score(y_u, y_pred)
    return acc, bacc, y_u.tolist(), y_pred.tolist(), info


print("Funzione valutazione pronta")

In [ ]:
# ============================================================
# TRAINING — tutti i soggetti, entrambi i modelli
# ============================================================

subjects = sorted(meta["subject_id"].unique())
print(f"Soggetti: {len(subjects)}")
print(f"Chance level: {100/N_CLASSES:.1f}%")
print()

results_all = []

for model_name in ["DHSLP", "DHSLF"]:
    print(f"
{'='*50}")
    print(f"  Modello: {model_name}")
    print(f"{'='*50}")

    accs, baccs = [], []
    ys_all, ps_all = [], []

    for subj in tqdm(subjects, desc=model_name):
        records = meta[meta["subject_id"] == subj][
            ["path_h5", "epoch_idx", "label_idx"]
        ].to_dict("records")

        result = evaluate_subject(records, keep_idx, model=model_name)
        if result is None:
            continue

        acc, bacc, ys, ps, info = result
        accs.append(acc)
        baccs.append(bacc)
        ys_all.extend(ys)
        ps_all.extend(ps)

    mean_acc  = np.mean(accs)
    mean_bacc = np.mean(baccs)
    overall_bacc = balanced_accuracy_score(ys_all, ps_all)

    print(f"  mean acc (per sogg):    {mean_acc:.4f}")
    print(f"  mean bacc (per sogg):   {mean_bacc:.4f}")
    print(f"  overall bacc (pooled):  {overall_bacc:.4f}")
    print(f"  chance level:           {1/N_CLASSES:.4f}")

    results_all.append({
        "model":         model_name,
        "mean_acc":      mean_acc,
        "mean_bacc":     mean_bacc,
        "overall_bacc":  overall_bacc,
        "n_subjects":    len(accs),
    })

df_results = pd.DataFrame(results_all)
df_results.to_csv(RESULTS_CSV, index=False)
print(f"
Risultati salvati: {RESULTS_CSV}")

In [ ]:
# ============================================================
# CONFUSION MATRICES + CONFRONTO CON EEG_09/10
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_name in zip(axes, ["DHSLP", "DHSLF"]):
    ys_all, ps_all = [], []
    for subj in tqdm(subjects, desc=f"CM {model_name}", leave=False):
        records = meta[meta["subject_id"] == subj][
            ["path_h5", "epoch_idx", "label_idx"]
        ].to_dict("records")
        result = evaluate_subject(records, keep_idx, model=model_name)
        if result:
            _, _, ys, ps, _ = result
            ys_all.extend(ys)
            ps_all.extend(ps)

    cm   = confusion_matrix(ys_all, ps_all, normalize="true",
                            labels=list(range(N_CLASSES)))
    bacc = balanced_accuracy_score(ys_all, ps_all)

    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=cluster_names, yticklabels=cluster_names,
                ax=ax, vmin=0, vmax=1)
    ax.set_title(f"{model_name}  (within-subject, {LABEL_RATIO*100:.0f}% labeled)
"                 f"overall bacc = {bacc:.3f}  |  chance = {1/N_CLASSES:.3f}", fontsize=11)
    ax.set_xlabel("Predetto")
    ax.set_ylabel("Vero")

plt.tight_layout()
fig_path = project_root / "figures" / f"eeg11_dhslp_confusion_{N_CLASSES}cls.png"
fig_path.parent.mkdir(exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# ============================================================
# SOMMARIO FINALE — confronto con tutti i modelli precedenti
# ============================================================

print("
" + "="*60)
print("  RISULTATI FINALI — EEG_11 DHSLP/DHSLF")
print("="*60)
print(df_results[["model","mean_bacc","overall_bacc"]].to_string(index=False))

# Confronto cross-esperimento
chance = 1.0 / N_CLASSES
print(f"
Chance level: {chance:.4f} ({100*chance:.1f}%)")

# Carica risultati precedenti se disponibili
for csv_file, exp_name in [
    (project_root / "data" / "interim" / f"eeg09_gat_{N_CLASSES}_results.csv", "EEG_09 GAT"),
    (project_root / "data" / "interim" / f"eeg10_hgnn_{N_CLASSES}_results.csv", "EEG_10 HGNN"),
]:
    if csv_file.exists():
        df_prev = pd.read_csv(csv_file)
        for _, row in df_prev.iterrows():
            metric = row.get("test_bacc", row.get("overall_bacc", 0))
            print(f"  {exp_name} {row['model']:<20s}: {metric:.4f}  (subject-independent)")

for _, row in df_results.iterrows():
    delta = row["overall_bacc"] - chance
    print(f"  EEG_11 {row['model']:<24s}: {row['overall_bacc']:.4f}  (+{delta:.4f} vs chance) [within-subj]")